In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# ============================================================
# Projected annual exceedance-day curves (2 × 2: rows = wave/wind, columns = SSP1-2.6 / SSP5-8.5)
# Input: C2_hazard_annual_series.csv (produced by vis_pro_extreme_wind_wave.ipynb cell 0)
#   columns: year, cut, value, base, delta, pct, variable, model, scenario
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CSV = rf'{DATA_ROOT}/new_climate/_proj_main/tables/C2_hazard_annual_series.csv'
TS  = pd.read_csv(CSV)

# ── Tunable parameters ────────────────────────────────────────────────
VALUE      = 'pct'          # 'pct' = % relative to ERA5 baseline  |  'delta' = d yr⁻¹
ROLL       = 10             # centred moving-average window (years)
MIN_PER    = ROLL // 2      # minimum number of years required at the ends
START, END = 2020, 2100     # x-axis range (data start in 2015; the moving average uses 2015-2019)
LATE       = (2081, 2100)   # shaded period (same as the seasonal heat map)
YLIM       = (-30, 5)       # None → automatic from data
SHOW_RAW   = False          # True: overlay raw annual values as thin light lines
STYLE      = 'model'        # 'model' = solid/dashed lines distinguish models (as in reference figure)
                            # 'band'  = thick two-model mean line + inter-model range shading
KEEP   = {'wave': [1.25, 2.5, 4.0, 6.0], 'wind': [8.0, 10.8, 13.9, 17.2]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']

# ── Labels / style ─────────────────────────────────────────────
MLAB  = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
SLAB  = {'ssp126': 'SSP1–2.6', 'ssp585': 'SSP5–8.5'}
LS    = {'ACCESS': '-', 'ECEARTH': (0, (6, 3))}
VLAB  = {'wave': 'Significant wave height', 'wind': '10-m wind speed'}
COL   = {'wave': ['#bfd8ee', '#7fb6d6', '#3f8fc1', '#0b5aa6'],     # light→dark = increasing threshold
         'wind': ['#fdc58f', '#f79249', '#e85a13', '#a93400']}
INK, INK2, GRID, ZERO, SHADE = '#1a1a1a', '#5c6166', '#dedede', '#9b9b9b', '#f4f4f4'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'mathtext.fontset': 'dejavusans',
                     'axes.linewidth': 0.85})

# ── Helper functions ────────────────────────────────────────────────
def series(kind, cut, ssp, model):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut) &
           (TS.scenario == ssp) & (TS.model == model)]
    return d.set_index('year')[VALUE].astype(float).sort_index()

def smooth(s):
    return s.rolling(ROLL, center=True, min_periods=MIN_PER).mean()

def baseline(kind, cut):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut)]['base'].dropna()
    return float(d.iloc[0]) if len(d) else np.nan

def thr_label(kind, cut):
    lhs = (rf'$H_s \geq {cut:g}$ m' if kind == 'wave'
           else rf'$U_{{10}} \geq {cut:g}$ m s$^{{-1}}$')
    b = baseline(kind, cut)
    return f'{lhs}    ({b:.0f} d yr$^{{-1}}$)' if np.isfinite(b) else lhs

def auto_ylim():
    v = TS[TS.year.between(START, END) & TS.model.isin(MODELS) &
           TS.scenario.isin(SSPS)][VALUE].replace([np.inf, -np.inf], np.nan).dropna()
    lo = 5 * np.floor(min(v.quantile(.01), 0) / 5) - 5
    hi = max(5, 5 * np.ceil(max(v.quantile(.99), 0) / 5))
    return lo, hi

# ── Plot ────────────────────────────────────────────────────
ylo, yhi = YLIM if YLIM is not None else auto_ylim()
fig = plt.figure(figsize=(15.6, 8.9), dpi=150)
gs  = fig.add_gridspec(2, 2, left=.115, right=.785, top=.845, bottom=.145,
                       wspace=.10, hspace=.19)
axes = np.empty((2, 2), dtype=object)

for i, kind in enumerate(['wave', 'wind']):
    for j, ssp in enumerate(SSPS):
        ax = axes[i, j] = fig.add_subplot(gs[i, j])
        ax.axvspan(*LATE, color=SHADE, lw=0, zorder=0)
        ax.axhline(0, color=ZERO, lw=.9, zorder=1)
        ax.grid(axis='y', color=GRID, lw=.65, alpha=.75, zorder=0)

        for k, cut in enumerate(KEEP[kind]):
            c = COL[kind][k]
            sm = {m: smooth(series(kind, cut, ssp, m)) for m in MODELS}
            if STYLE == 'model':
                for m in MODELS:
                    if SHOW_RAW:
                        raw = series(kind, cut, ssp, m)
                        ax.plot(raw.index, raw.values, color=c, ls=LS[m],
                                lw=.6, alpha=.25, zorder=2)
                    ax.plot(sm[m].index, sm[m].values, color=c, ls=LS[m], lw=2.15,
                            solid_capstyle='round', dash_capstyle='round', zorder=4)
            else:                                   # 'band'
                M = pd.concat(sm, axis=1)
                ax.fill_between(M.index, M.min(1), M.max(1), color=c, alpha=.22,
                                lw=0, zorder=2)
                ax.plot(M.index, M.mean(1), color=c, lw=2.3, zorder=4)

        ax.set_xlim(START, END); ax.set_ylim(ylo, yhi)
        ax.set_xticks(np.arange(START, END + 1, 20))
        step = 10 if (yhi - ylo) >= 30 else 5
        ax.set_yticks(np.arange(np.ceil(ylo / step) * step, yhi + .1, step))
        ax.tick_params(labelsize=10, colors=INK, length=4, pad=4)
        for sp in ('top', 'right'):
            ax.spines[sp].set_visible(False)
        if i == 0:
            ax.set_title(SLAB[ssp], fontsize=14, fontweight='bold', color=INK, pad=12)
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel('Year', fontsize=11.5, color=INK, labelpad=6)
        if j == 1:
            ax.tick_params(labelleft=False)
            ax.text(np.mean(LATE), yhi - .045 * (yhi - ylo), f'{LATE[0]}–{LATE[1]}',
                    ha='center', va='top', fontsize=8.5, color='#777777', zorder=5)

    p = axes[i, 0].get_position()
    fig.text(.053, p.y0 + p.height / 2, VLAB[kind], rotation=90, va='center',
             ha='center', fontsize=11.5, fontweight='bold', color=INK)

# Shared y label, title, subtitle
ylab = ('Change in annual threshold-exceedance days (%)' if VALUE == 'pct'
        else r'Change in annual threshold-exceedance days (d yr$^{-1}$)')
fig.text(.020, .49, ylab, rotation=90, va='center', ha='center', fontsize=11, color=INK)
fig.text(.060, .952, 'a', fontsize=17, fontweight='bold', color=INK)
fig.text(.115, .952, 'Projected change in annual threshold-exceedance days',
         fontsize=17, fontweight='bold', color=INK)
sub = ('Relative to ERA5 (2002–2020); five regions combined using traffic weights · '
       f'{ROLL}-year centred running mean')
if STYLE == 'band':
    sub += ' · line = two-model mean, shading = inter-model range'
fig.text(.115, .918, sub, fontsize=10.5, color=INK2)

# Right-hand legend: models (STYLE='model' only) + wave thresholds + wind thresholds
def side_legend(handles, title, y, **kw):
    lg = fig.legend(handles=handles, title=title, loc='upper left',
                    bbox_to_anchor=(.812, y), frameon=False, fontsize=9.9,
                    title_fontsize=10.5, borderaxespad=0, **kw)
    lg._legend_box.align = 'left'
    fig.add_artist(lg)

if STYLE == 'model':
    side_legend([Line2D([0], [0], color='#222222', lw=2.1, ls=LS[m], label=MLAB[m])
                 for m in MODELS], 'Climate model', .905,
                handlelength=3.1, handletextpad=.8)
side_legend([Line2D([0], [0], color=COL['wave'][k], lw=2.7, label=thr_label('wave', c))
             for k, c in enumerate(KEEP['wave'])], 'Wave threshold   (baseline)', .735,
            handlelength=2.9, handletextpad=.9, labelspacing=.85)
side_legend([Line2D([0], [0], color=COL['wind'][k], lw=2.7, label=thr_label('wind', c))
             for k, c in enumerate(KEEP['wind'])], 'Wind threshold   (baseline)', .355,
            handlelength=2.9, handletextpad=.9, labelspacing=.85)

fig.text(.115, .055, f'At projection edges, centred means use at least {MIN_PER} '
         'available annual values.', fontsize=8.5, color=INK2)
plt.show()


In [ ]:
# ============================================================
# Projected annual exceedance-day curves: unlabelled version, one (variable, scenario) panel per figure
#   no title, subtitle, legend, panel letters, shading text or row labels; only curves, axes and shading
#   data processing identical to the 2×2 version (baseline = each (model, ssp)'s own 2021–2040 mean)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV    = rf'{DATA_ROOT}/new_climate/_proj_main/tables/C2_hazard_annual_series.csv'
REBASE = True
REF_Y  = (2021, 2040)

# ── Tunable parameters ────────────────────────────────────────────────
VALUE      = 'pct'
ROLL       = 20
MIN_PER    = ROLL // 2
START, END = 2021, 2100
LATE       = (2081, 2100)
YLIM       = (-25, 10)              # None → automatic from all panel data (shared by the four figures)
SHOW_RAW   = False
STYLE      = 'model'                # 'model' | 'band'
SHADING    = True                   # whether to shade reference period / end of century
AXIS_LABEL = False                  # True: keep Year and y-axis text; False: ticks only
TICK_LABEL = True                   # False: drop tick numbers as well
FIGSIZE    = (6.0, 4.0)
DPI        = 500
KEEP   = {'wave': [1.25, 2.5, 4.0, 6.0], 'wind': [8.0, 10.8, 13.9, 17.2]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']
PANELS = [(k, s) for k in ('wave', 'wind') for s in SSPS]     # figure order; entries can be removed

# ── Style (same as in M4)────────────────────────────────
LS   = {'ACCESS': '-', 'ECEARTH': (0, (6, 3))}
COL  = {'wave': ['#c8dcee', '#7fb6d6', '#3f8fc1', '#0b5aa6'],
        'wind': ['#fdd4ad', '#f79249', '#e85a13', '#a93400']}
INK, GRID, ZERO = '#1a1a1a', '#dedede', '#9b9b9b'
SHADE_LATE, SHADE_REF = '#f4f4f4', '#eaf0f7'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'mathtext.fontset': 'dejavusans',
                     'axes.linewidth': 0.85})

# ── Read table + re-base ─────────────────────────────────────────
TS = pd.read_csv(CSV)
if REBASE:
    key = ['variable', 'cut', 'model', 'scenario']
    base = (TS[TS.year.between(*REF_Y)].groupby(key).value.mean()
              .rename('base').reset_index())
    TS = TS.drop(columns=['base', 'delta', 'pct'], errors='ignore').merge(base, on=key)
    TS['delta'] = TS.value - TS.base
    TS['pct']   = TS.delta / TS.base * 100

def series(kind, cut, ssp, model):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut) &
           (TS.scenario == ssp) & (TS.model == model)]
    return d.set_index('year')[VALUE].astype(float).sort_index()

def smooth(s):
    return s.rolling(ROLL, center=True, min_periods=MIN_PER).mean()

def auto_ylim():
    v = TS[TS.year.between(START, END) & TS.model.isin(MODELS) & TS.scenario.isin(SSPS)
           & (TS.cut.isin(KEEP['wave']) | TS.cut.isin(KEEP['wind']))][VALUE]
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    lo = 5 * np.floor(min(v.quantile(.01), 0) / 5) - 5
    hi = max(5, 5 * np.ceil(max(v.quantile(.99), 0) / 5))
    return lo, hi

# ── Single panel ──────────────────────────────────────────────────
def plot_panel(kind, ssp):
    ylo, yhi = YLIM if YLIM is not None else auto_ylim()
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    fig.subplots_adjust(left=.13, right=.97, top=.96, bottom=.14)

    if SHADING:
        ax.axvspan(*REF_Y, color=SHADE_REF, lw=0, zorder=0)
        ax.axvspan(*LATE,  color=SHADE_LATE, lw=0, zorder=0)
    ax.axhline(0, color=ZERO, lw=.9, zorder=1)
    ax.grid(axis='y', color=GRID, lw=.65, alpha=.75, zorder=0)

    for k, cut in enumerate(KEEP[kind]):
        c = COL[kind][k]
        sm = {m: smooth(series(kind, cut, ssp, m)) for m in MODELS}
        if STYLE == 'model':
            for m in MODELS:
                if SHOW_RAW:
                    raw = series(kind, cut, ssp, m)
                    ax.plot(raw.index, raw.values, color=c, ls=LS[m], lw=.6, alpha=.25, zorder=2)
                ax.plot(sm[m].index, sm[m].values, color=c, ls=LS[m], lw=2.15,
                        solid_capstyle='round', dash_capstyle='round', zorder=4)
        else:
            M = pd.concat(sm, axis=1)
            ax.fill_between(M.index, M.min(1), M.max(1), color=c, alpha=.22, lw=0, zorder=2)
            ax.plot(M.index, M.mean(1), color=c, lw=2.3, zorder=4)

    ax.set_xlim(START, END); ax.set_ylim(ylo, yhi)
    ax.set_xticks([2021, 2040, 2060, 2080, 2100])
    step = 10 if (yhi - ylo) >= 30 else 5
    ax.set_yticks(np.arange(np.ceil(ylo / step) * step, yhi + .1, step))
    ax.tick_params(labelsize=10, colors=INK, length=4, pad=4,
                   labelbottom=TICK_LABEL, labelleft=TICK_LABEL)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    if AXIS_LABEL:
        ax.set_xlabel('Year', fontsize=11, color=INK)
        ax.set_ylabel('Change in annual threshold-exceedance days (%)' if VALUE == 'pct'
                      else r'Change in annual threshold-exceedance days (d yr$^{-1}$)',
                      fontsize=10, color=INK)
    plt.show()

# ── One figure at a time (order: wave ssp126, wave ssp585, wind ssp126, wind ssp585)──
for kind, ssp in PANELS:
    print(f'--- {kind} {ssp}')
    plot_panel(kind, ssp)


In [ ]:
# ============================================================
# Projected annual exceedance-day curves: one (variable, SSP) combination per figure, plt.show() each
# Input: C2_hazard_annual_series.csv (traffic-weighted five-region aggregate, not global)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CSV = rf'{DATA_ROOT}/new_climate/_proj_main/tables/C2_hazard_annual_series.csv'
TS  = pd.read_csv(CSV)

# ── Tunable parameters ────────────────────────────────────────────────
VALUE      = 'pct'          # 'pct' = %  |  'delta' = d yr⁻¹
ROLL       = 10             # centred moving-average window (years)
MIN_PER    = ROLL // 2
START, END = 2020, 2100
LATE       = (2081, 2100)
YLIM       = (-30, 5)       # None → automatic per figure
SHOW_RAW   = False          # True: overlay raw annual values as thin light lines
STYLE      = 'model'        # 'model' solid/dashed per model | 'band' two-model mean + range band
KEEP   = {'wave': [1.25, 2.5, 4.0, 6.0], 'wind': [8.0, 10.8, 13.9, 17.2]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']
PANELS = [(k, s) for k in ('wave', 'wind') for s in SSPS]   # figure order; entries can be removed
LETTERS = 'abcdefgh'

# ── Labels / style ─────────────────────────────────────────────
MLAB = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
SLAB = {'ssp126': 'SSP1–2.6', 'ssp585': 'SSP5–8.5'}
LS   = {'ACCESS': '-', 'ECEARTH': (0, (6, 3))}
VLAB = {'wave': 'Significant wave height', 'wind': '10-m wind speed'}
COL  = {'wave': ['#bfd8ee', '#7fb6d6', '#3f8fc1', '#0b5aa6'],
        'wind': ['#fdc58f', '#f79249', '#e85a13', '#a93400']}
INK, INK2, GRID, ZERO, SHADE = '#1a1a1a', '#5c6166', '#dedede', '#9b9b9b', '#f4f4f4'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'mathtext.fontset': 'dejavusans',
                     'axes.linewidth': 0.85})

# ── Helper functions ────────────────────────────────────────────────
def series(kind, cut, ssp, model):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut) &
           (TS.scenario == ssp) & (TS.model == model)]
    return d.set_index('year')[VALUE].astype(float).sort_index()

def smooth(s):
    return s.rolling(ROLL, center=True, min_periods=MIN_PER).mean()

def baseline(kind, cut):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut)]['base'].dropna()
    return float(d.iloc[0]) if len(d) else np.nan

def thr_label(kind, cut):
    lhs = (rf'$H_s \geq {cut:g}$ m' if kind == 'wave'
           else rf'$U_{{10}} \geq {cut:g}$ m s$^{{-1}}$')
    b = baseline(kind, cut)
    return f'{lhs}   ({b:.0f} d yr$^{{-1}}$)' if np.isfinite(b) else lhs

def auto_ylim(kind, ssp):
    v = TS[(TS.variable == kind) & (TS.scenario == ssp) & TS.model.isin(MODELS) &
           TS.cut.isin(KEEP[kind]) & TS.year.between(START, END)][VALUE]
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    lo = 5 * np.floor(min(v.quantile(.01), 0) / 5) - 5
    hi = max(5, 5 * np.ceil(max(v.quantile(.99), 0) / 5))
    return lo, hi

# ── Single-figure function ────────────────────────────────────────────────
def plot_one(kind, ssp, letter=None):
    ylo, yhi = YLIM if YLIM is not None else auto_ylim(kind, ssp)
    fig, ax = plt.subplots(figsize=(8.6, 5.2), dpi=150)
    fig.subplots_adjust(left=.11, right=.70, top=.83, bottom=.14)

    ax.axvspan(*LATE, color=SHADE, lw=0, zorder=0)
    ax.axhline(0, color=ZERO, lw=.9, zorder=1)
    ax.grid(axis='y', color=GRID, lw=.65, alpha=.75, zorder=0)

    for k, cut in enumerate(KEEP[kind]):
        c = COL[kind][k]
        sm = {m: smooth(series(kind, cut, ssp, m)) for m in MODELS}
        if STYLE == 'model':
            for m in MODELS:
                if SHOW_RAW:
                    raw = series(kind, cut, ssp, m)
                    ax.plot(raw.index, raw.values, color=c, ls=LS[m], lw=.6,
                            alpha=.25, zorder=2)
                ax.plot(sm[m].index, sm[m].values, color=c, ls=LS[m], lw=2.15,
                        solid_capstyle='round', dash_capstyle='round', zorder=4)
        else:
            M = pd.concat(sm, axis=1)
            ax.fill_between(M.index, M.min(1), M.max(1), color=c, alpha=.22,
                            lw=0, zorder=2)
            ax.plot(M.index, M.mean(1), color=c, lw=2.3, zorder=4)

    ax.set_xlim(START, END); ax.set_ylim(ylo, yhi)
    ax.set_xticks(np.arange(START, END + 1, 20))
    step = 10 if (yhi - ylo) >= 30 else 5
    ax.set_yticks(np.arange(np.ceil(ylo / step) * step, yhi + .1, step))
    ax.tick_params(labelsize=10, colors=INK, length=4, pad=4)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    ax.set_xlabel('Year', fontsize=11, color=INK)
    ax.set_ylabel('Change in annual threshold-exceedance days (%)' if VALUE == 'pct'
                  else r'Change in annual threshold-exceedance days (d yr$^{-1}$)',
                  fontsize=10.5, color=INK)
    ax.text(np.mean(LATE), yhi - .045 * (yhi - ylo), f'{LATE[0]}–{LATE[1]}',
            ha='center', va='top', fontsize=8.5, color='#777777', zorder=5)

    # Title
    if letter:
        fig.text(.03, .94, letter, fontsize=15, fontweight='bold', color=INK)
    fig.text(.11, .94, f'{VLAB[kind]} · {SLAB[ssp]}', fontsize=14,
             fontweight='bold', color=INK)
    sub = ('Relative to ERA5 (2002–2020); five regions combined using traffic weights · '
           f'{ROLL}-yr centred running mean')
    if STYLE == 'band':
        sub += ' · line = two-model mean, shading = inter-model range'
    fig.text(.11, .895, sub, fontsize=8.5, color=INK2)

    # Right-hand legend
    def side_legend(handles, title, y, **kw):
        lg = fig.legend(handles=handles, title=title, loc='upper left',
                        bbox_to_anchor=(.72, y), frameon=False, fontsize=9.5,
                        title_fontsize=10, borderaxespad=0, **kw)
        lg._legend_box.align = 'left'
        fig.add_artist(lg)

    y_thr = .83
    if STYLE == 'model':
        side_legend([Line2D([0], [0], color='#222222', lw=2.1, ls=LS[m], label=MLAB[m])
                     for m in MODELS], 'Climate model', .83,
                    handlelength=3.0, handletextpad=.8)
        y_thr = .62
    ttl = 'Wave threshold   (baseline)' if kind == 'wave' else 'Wind threshold   (baseline)'
    side_legend([Line2D([0], [0], color=COL[kind][k], lw=2.7, label=thr_label(kind, c))
                 for k, c in enumerate(KEEP[kind])], ttl, y_thr,
                handlelength=2.8, handletextpad=.9, labelspacing=.8)

    fig.text(.11, .035, f'At projection edges, centred means use at least {MIN_PER} '
             'available annual values.', fontsize=8, color=INK2)
    plt.show()

# ── One figure at a time ────────────────────────────────────────────────
for n, (kind, ssp) in enumerate(PANELS):
    plot_one(kind, ssp, letter=LETTERS[n])


In [ ]:
# ============================================================
# C2 v2: annual exceedance-day series, baseline = each (model, ssp)'s own 2021–2040 mean
# Output: C2_hazard_annual_series_ref2021_2040.csv
# columns: year, cut, value, base, delta, pct, variable, model, scenario, ref
# ============================================================
import os, numpy as np, pandas as pd, xarray as xr

O   = rf'{DATA_ROOT}/new_climate/_proj_main'
TB  = os.path.join(O, 'tables')
M2C = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid2'

REF_Y  = (2021, 2040)                     # ★ reference period, computed per (model, ssp)
CUTS   = {'wave': [1.25, 2.5, 4.0, 6.0, 9.0], 'wind': [8.0, 10.8, 13.9, 17.2, 20.8]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']
REG = {'NW_Europe':   [(36,45,-15,-6),(45,63,-15,-6),(43,63,-6,0),(48,63,0,13)],
       'NW_Atlantic': [(18,31,-98,-60),(31,42,-82,-60),(42,50,-67,-60),(45,55,-60,-46)],
       'Med_Black':   [(30,43,-6,0),(30,46,0,19),(30,46,19,26.5),(30,41,26.5,37),(41,47.5,26.5,42)],
       'Asia':        [(-10,20,95,130),(20,46,117,146)],
       'Indian_Ocean_N': [(5,30,50,78),(5,26,78,95)]}

# ── cell list, region membership, GOOD mask, traffic weights (same as old cell 0)──
cells = xr.open_dataarray(os.path.join(O, 'obs_wave.nc')).cell.values
t0 = pd.to_datetime(xr.open_dataarray(os.path.join(O, 'cm_wave_ACCESS_his.nc')).time.values)
k0 = (t0.year >= 2002) & (t0.year <= 2014)
GOOD = np.ones(len(cells), bool)
for m in MODELS:
    for kd in ('wave', 'wind'):
        GOOD &= np.isfinite(xr.open_dataarray(
            os.path.join(O, f'cm_{kd}_{m}_his.nc')).load().values[k0]).mean(0) >= 0.9

def reg_of(g):
    la, lo = map(float, g.split('_'))
    for r, bs in REG.items():
        for S, N, W, E in bs:
            if S <= la < N and W <= lo < E:
                return r
    return None
RG = np.array([reg_of(g) for g in cells])

pg = pd.read_parquet(os.path.join(M2C, 'panel_global.parquet'),
                     columns=['region', 'grid_id', 'E_it'])
pg = pg[pg.grid_id.isin(set(cells[GOOD]))]
WR = pg.groupby('region').E_it.sum(); WR = WR / WR.sum()      # five-region traffic weights
print('cells usable', int(GOOD.sum()), '/', len(cells))
print('region weights:', WR.round(3).to_dict())

def annual_agg(arr, times, kind):
    """(T, NCELL) → DataFrame[year, cut, value]: cell mean within region → traffic-weighted over five regions"""
    yr = times.year.values; u = np.unique(yr); idx = np.searchsorted(yr, u)
    ok = np.isfinite(arr); out = []
    for c in CUTS[kind]:
        cnt = np.add.reduceat((ok & (arr >= c)).astype(np.float32), idx, 0)
        v = sum(cnt[:, (RG == r) & GOOD].mean(1) * WR[r] for r in REG)
        out.append(pd.DataFrame({'year': u, 'cut': c, 'value': v}))
    return pd.concat(out, ignore_index=True)

# ── Main loop: each (kind, model, ssp) uses its own 2021–2040 as baseline ──
rows = []
for kind in ('wave', 'wind'):
    for m in MODELS:
        for ssp in SSPS:
            q = xr.open_dataarray(os.path.join(O, f'qdm_{kind}_{m}_{ssp}.nc')).load()
            d = annual_agg(q.values, pd.to_datetime(q.time.values), kind)
            base = d[d.year.between(*REF_Y)].groupby('cut').value.mean()   # ★
            d['base']  = d.cut.map(base)
            d['delta'] = d.value - d.base
            d['pct']   = d.delta / d.base * 100
            d['variable'] = kind; d['model'] = m; d['scenario'] = ssp
            d['ref'] = f'{REF_Y[0]}-{REF_Y[1]}'
            rows.append(d); del q
            print(f'  {kind} {m} {ssp}  base:',
                  {c: round(float(base[c]), 1) for c in CUTS[kind]}, flush=True)

TS = pd.concat(rows, ignore_index=True)
out = os.path.join(TB, 'C2_hazard_annual_series_ref2021_2040.csv')
TS.to_csv(out, index=False)
print('\nC2 v2', TS.shape, '→', out)

# Self-check: mean pct within the reference period should be 0; also inspect end of century
chk = (TS[TS.year.between(*REF_Y)].groupby(['variable', 'model', 'scenario']).pct.mean())
print('\nreference-period mean pct (should be ≈0):\n', chk.round(6).to_string())
late = (TS[TS.year.between(2081, 2100) & TS.cut.isin([6.0, 17.2])]
        .groupby(['variable', 'cut', 'scenario', 'model']).pct.mean())
print('\n2081–2100 relative to 2021–2040 (%), H_s≥6 / U10≥17.2:\n', late.round(1).to_string())


In [ ]:
# ============================================================
# Projected annual exceedance-day curves (2 × 2), baseline = each (model, ssp)'s own 2021–2040 mean
# Input: C2_hazard_annual_series.csv (old table is fine: value column holds absolute days, re-based here)
#      if C2_hazard_annual_series_ref2021_2040.csv already exists, change the CSV path and set REBASE=False
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CSV    = rf'{DATA_ROOT}/new_climate/_proj_main/tables/C2_hazard_annual_series.csv'
REBASE = True                       # True: recompute base/delta/pct from the value column using REF_Y
REF_Y  = (2021, 2040)               # ★ reference period, computed per (model, ssp)

# ── Tunable parameters ────────────────────────────────────────────────
VALUE      = 'pct'                  # 'pct' = %  |  'delta' = d yr⁻¹
ROLL       = 20                     # centred moving-average window (years)
MIN_PER    = ROLL // 2
START, END = 2021, 2100
LATE       = (2081, 2100)           # end-of-century shading
YLIM       = (-25, 10)              # None → automatic from data
SHOW_RAW   = False                  # True: overlay raw annual values as thin light lines
STYLE      = 'model'                # 'model' solid/dashed per model | 'band' two-model mean + range band
KEEP   = {'wave': [1.25, 2.5, 4.0, 6.0], 'wind': [8.0, 10.8, 13.9, 17.2]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']

# ── Read table + re-base ─────────────────────────────────────────
TS = pd.read_csv(CSV)
if REBASE:
    key = ['variable', 'cut', 'model', 'scenario']
    base = (TS[TS.year.between(*REF_Y)].groupby(key).value.mean()
              .rename('base').reset_index())
    TS = TS.drop(columns=['base', 'delta', 'pct'], errors='ignore').merge(base, on=key)
    TS['delta'] = TS.value - TS.base
    TS['pct']   = TS.delta / TS.base * 100
    TS['ref']   = f'{REF_Y[0]}-{REF_Y[1]}'

# ── Labels / style ─────────────────────────────────────────────
MLAB = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
SLAB = {'ssp126': 'SSP1–2.6', 'ssp585': 'SSP5–8.5'}
LS   = {'ACCESS': '-', 'ECEARTH': (0, (6, 3))}
VLAB = {'wave': 'Significant wave height', 'wind': '10-m wind speed'}
COL  = {'wave': ["#c8dcee", '#7fb6d6', '#3f8fc1', '#0b5aa6'],     # light→dark = increasing threshold
        'wind': ["#fdd4ad", '#f79249', '#e85a13', '#a93400']}
INK, INK2, GRID, ZERO = '#1a1a1a', '#5c6166', '#dedede', '#9b9b9b'
SHADE_LATE, SHADE_REF = '#f4f4f4', '#eaf0f7'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'mathtext.fontset': 'dejavusans',
                     'axes.linewidth': 0.85})

# ── Helper functions ────────────────────────────────────────────────
def series(kind, cut, ssp, model):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut) &
           (TS.scenario == ssp) & (TS.model == model)]
    return d.set_index('year')[VALUE].astype(float).sort_index()

def smooth(s):
    return s.rolling(ROLL, center=True, min_periods=MIN_PER).mean()

def baseline(kind, cut):
    """Mean of the baselines of the four (model, ssp) combinations, used only for legend labels"""
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut)]
    return float(d.groupby(['model', 'scenario']).base.first().mean())

def thr_label(kind, cut):
    lhs = (rf'$H_s \geq {cut:g}$ m' if kind == 'wave'
           else rf'$U_{{10}} \geq {cut:g}$ m s$^{{-1}}$')
    return f'{lhs}    (~{baseline(kind, cut):.0f} d yr$^{{-1}}$)'

def auto_ylim():
    v = TS[TS.year.between(START, END) & TS.model.isin(MODELS) & TS.scenario.isin(SSPS)
           & (TS.cut.isin(KEEP['wave']) | TS.cut.isin(KEEP['wind']))][VALUE]
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    lo = 5 * np.floor(min(v.quantile(.01), 0) / 5) - 5
    hi = max(5, 5 * np.ceil(max(v.quantile(.99), 0) / 5))
    return lo, hi

# ── Plot ────────────────────────────────────────────────────
ylo, yhi = YLIM if YLIM is not None else auto_ylim()
fig = plt.figure(figsize=(15.6, 8.9), dpi=500)
gs  = fig.add_gridspec(2, 2, left=.115, right=.785, top=.845, bottom=.145,
                       wspace=.10, hspace=.19)
axes = np.empty((2, 2), dtype=object)

for i, kind in enumerate(['wave', 'wind']):
    for j, ssp in enumerate(SSPS):
        ax = axes[i, j] = fig.add_subplot(gs[i, j])
        ax.axvspan(*REF_Y, color=SHADE_REF, lw=0, zorder=0)        # reference period
        ax.axvspan(*LATE,  color=SHADE_LATE, lw=0, zorder=0)       # end of century
        ax.axhline(0, color=ZERO, lw=.9, zorder=1)
        ax.grid(axis='y', color=GRID, lw=.65, alpha=.75, zorder=0)

        for k, cut in enumerate(KEEP[kind]):
            c = COL[kind][k]
            sm = {m: smooth(series(kind, cut, ssp, m)) for m in MODELS}
            if STYLE == 'model':
                for m in MODELS:
                    if SHOW_RAW:
                        raw = series(kind, cut, ssp, m)
                        ax.plot(raw.index, raw.values, color=c, ls=LS[m], lw=.6,
                                alpha=.25, zorder=2)
                    ax.plot(sm[m].index, sm[m].values, color=c, ls=LS[m], lw=2.15,
                            solid_capstyle='round', dash_capstyle='round', zorder=4)
            else:
                M = pd.concat(sm, axis=1)
                ax.fill_between(M.index, M.min(1), M.max(1), color=c, alpha=.22,
                                lw=0, zorder=2)
                ax.plot(M.index, M.mean(1), color=c, lw=2.3, zorder=4)

        ax.set_xlim(START, END); ax.set_ylim(ylo, yhi)
        ax.set_xticks([2021, 2040, 2060, 2080, 2100])
        step = 10 if (yhi - ylo) >= 30 else 5
        ax.set_yticks(np.arange(np.ceil(ylo / step) * step, yhi + .1, step))
        ax.tick_params(labelsize=10, colors=INK, length=4, pad=4)
        for sp in ('top', 'right'):
            ax.spines[sp].set_visible(False)
        if i == 0:
            ax.set_title(SLAB[ssp], fontsize=14, fontweight='bold', color=INK, pad=12)
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel('Year', fontsize=11.5, color=INK, labelpad=6)
        if j == 1:
            ax.tick_params(labelleft=False)
        # shading labels: first row only, to avoid repetition
        if i == 0:
            ytxt = yhi - .045 * (yhi - ylo)
            ax.text(np.mean(REF_Y), ytxt, 'reference', ha='center', va='top',
                    fontsize=8.5, color='#777777', zorder=5)
            ax.text(np.mean(LATE), ytxt, f'{LATE[0]}–{LATE[1]}', ha='center', va='top',
                    fontsize=8.5, color='#777777', zorder=5)

    p = axes[i, 0].get_position()
    fig.text(.053, p.y0 + p.height / 2, VLAB[kind], rotation=90, va='center',
             ha='center', fontsize=11.5, fontweight='bold', color=INK)

# Shared y label, title, subtitle
ylab = ('Change in annual threshold-exceedance days (%)' if VALUE == 'pct'
        else r'Change in annual threshold-exceedance days (d yr$^{-1}$)')
fig.text(.020, .49, ylab, rotation=90, va='center', ha='center', fontsize=11, color=INK)
fig.text(.060, .952, 'a', fontsize=17, fontweight='bold', color=INK)
fig.text(.115, .952, 'Projected change in annual threshold-exceedance days',
         fontsize=17, fontweight='bold', color=INK)
sub = (f"Relative to each model–scenario's own {REF_Y[0]}–{REF_Y[1]} mean; "
       f"five regions combined using traffic weights · {ROLL}-year centred running mean")
if STYLE == 'band':
    sub += ' · line = two-model mean, shading = inter-model range'
fig.text(.115, .918, sub, fontsize=10.5, color=INK2)

# Right-hand legend
def side_legend(handles, title, y, **kw):
    lg = fig.legend(handles=handles, title=title, loc='upper left',
                    bbox_to_anchor=(.812, y), frameon=False, fontsize=9.9,
                    title_fontsize=10.5, borderaxespad=0, **kw)
    lg._legend_box.align = 'left'
    fig.add_artist(lg)

if STYLE == 'model':
    side_legend([Line2D([0], [0], color='#222222', lw=2.1, ls=LS[m], label=MLAB[m])
                 for m in MODELS], 'Climate model', .905,
                handlelength=3.1, handletextpad=.8)
ref_txt = f'({REF_Y[0]}–{REF_Y[1]} mean)'
side_legend([Line2D([0], [0], color=COL['wave'][k], lw=2.7, label=thr_label('wave', c))
             for k, c in enumerate(KEEP['wave'])], f'Wave threshold   {ref_txt}', .735,
            handlelength=2.9, handletextpad=.9, labelspacing=.85)
side_legend([Line2D([0], [0], color=COL['wind'][k], lw=2.7, label=thr_label('wind', c))
             for k, c in enumerate(KEEP['wind'])], f'Wind threshold   {ref_txt}', .355,
            handlelength=2.9, handletextpad=.9, labelspacing=.85)

fig.text(.115, .055,
         f'Baseline days shown are the mean of the four model–scenario baselines. '
         f'At projection edges, centred means use at least {MIN_PER} annual values.',
         fontsize=8.5, color=INK2)
plt.show()


In [ ]:
# ============================================================
# Projected annual exceedance-day curves (2 × 2: rows = wave/wind, columns = SSP1-2.6 / SSP5-8.5), four thresholds per panel
# Three legibility measures:
#   1. PALETTE='categorical': four hues for the four thresholds (blue/green/orange/purple; verified: normal vision ΔE 16.3,
#      colour-vision deficiency ΔE 9.2, all pass); 'ordinal' uses a re-graded single-hue ramp (wider spread than the original)
#   2. line width increases with threshold, a "severity" cue in addition to hue
#   3. each threshold curve labelled directly at its right end, no need to consult the legend
# Baseline = each (model, ssp)'s own 2021–2040 mean (value column re-based directly)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CSV    = rf'{DATA_ROOT}/new_climate/_proj_main/tables/C2_hazard_annual_series.csv'
REBASE = True
REF_Y  = (2021, 2040)

# ── Tunable parameters ────────────────────────────────────────────────
VALUE      = 'pct'
ROLL       = 20
MIN_PER    = ROLL // 2
START, END = 2021, 2100
LATE       = (2081, 2100)
YLIM       = (-25, 10)          # None → automatic
STYLE      = 'model'            # 'model' = solid/dashed per model (8 lines) | 'band' = two-model mean + range band (4 lines)
PALETTE    = 'ordinal'      # 'categorical' | 'ordinal'
END_LABELS = True               # label thresholds directly at right end
KEEP   = {'wave': [1.25, 2.5, 4.0, 6.0], 'wind': [8.0, 10.8, 13.9, 17.2]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']

PALETTES = {
    'categorical': {'wave': ['#2a78d6', '#1baf7a', '#eb6834', '#4a3aa7'],
                    'wind': ['#2a78d6', '#1baf7a', '#eb6834', '#4a3aa7']},
    'ordinal':     {'wave': ['#7ec2e4', '#3190cb', '#155aa1', '#092858'],
                    'wind': ['#f6c053', '#ee7f24', '#cf3a17', '#6a1a05']},
}
COL = PALETTES[PALETTE]
LW  = [1.5, 1.9, 2.3, 2.8]      # line width increases with threshold
MLAB = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
SLAB = {'ssp126': 'SSP1–2.6', 'ssp585': 'SSP5–8.5'}
LS   = {'ACCESS': '-', 'ECEARTH': (0, (5, 2.5))}
VLAB = {'wave': 'Significant wave height', 'wind': '10-m wind speed'}
INK, INK2, GRID, ZERO = '#1a1a1a', '#5c6166', '#dedede', '#9b9b9b'
SHADE_LATE, SHADE_REF = '#f4f4f4', '#eaf0f7'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'mathtext.fontset': 'dejavusans',
                     'axes.linewidth': 0.85})

# ── Read table + re-base ─────────────────────────────────────────
TS = pd.read_csv(CSV)
if REBASE:
    key = ['variable', 'cut', 'model', 'scenario']
    base = (TS[TS.year.between(*REF_Y)].groupby(key).value.mean()
              .rename('base').reset_index())
    TS = TS.drop(columns=['base', 'delta', 'pct'], errors='ignore').merge(base, on=key)
    TS['delta'] = TS.value - TS.base
    TS['pct']   = TS.delta / TS.base * 100

# ── Helper functions ────────────────────────────────────────────────
def series(kind, cut, ssp, model):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut) &
           (TS.scenario == ssp) & (TS.model == model)]
    return d.set_index('year')[VALUE].astype(float).sort_index()

def smooth(s):
    return s.rolling(ROLL, center=True, min_periods=MIN_PER).mean()

def baseline(kind, cut):
    d = TS[(TS.variable == kind) & np.isclose(TS.cut, cut)]
    return float(d.groupby(['model', 'scenario']).base.first().mean())

def short_label(kind, cut):
    return f'≥{cut:g} m' if kind == 'wave' else f'≥{cut:g} m/s'

def thr_label(kind, cut):
    lhs = (rf'$H_s \geq {cut:g}$ m' if kind == 'wave'
           else rf'$U_{{10}} \geq {cut:g}$ m s$^{{-1}}$')
    return f'{lhs}    (~{baseline(kind, cut):.0f} d yr$^{{-1}}$)'

def auto_ylim():
    v = TS[TS.year.between(START, END) & TS.model.isin(MODELS) & TS.scenario.isin(SSPS)
           & (TS.cut.isin(KEEP['wave']) | TS.cut.isin(KEEP['wind']))][VALUE]
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    lo = 5 * np.floor(min(v.quantile(.01), 0) / 5) - 5
    hi = max(5, 5 * np.ceil(max(v.quantile(.99), 0) / 5))
    return lo, hi

def end_labels(ax, items, ylo, yhi):
    """items: [(y_end, color, text)] → staggered labels at right end"""
    items = sorted(items, key=lambda e: e[0])
    gap = .06 * (yhi - ylo); ys = []
    for y, c, txt in items:
        y = np.clip(y, ylo + gap / 2, yhi - gap / 2)
        if ys and y - ys[-1] < gap:
            y = ys[-1] + gap
        ys.append(y)
        ax.annotate(txt, (END, y), xytext=(5, 0), textcoords='offset points',
                    fontsize=8, color=c, va='center', ha='left', annotation_clip=False,
                    fontweight='bold')

# ── Plot ────────────────────────────────────────────────────
ylo, yhi = YLIM if YLIM is not None else auto_ylim()
fig = plt.figure(figsize=(15.6, 8.9), dpi=150)
gs  = fig.add_gridspec(2, 2, left=.115, right=.775, top=.845, bottom=.145,
                       wspace=.16, hspace=.19)
axes = np.empty((2, 2), dtype=object)

for i, kind in enumerate(['wave', 'wind']):
    for j, ssp in enumerate(SSPS):
        ax = axes[i, j] = fig.add_subplot(gs[i, j])
        ax.axvspan(*REF_Y, color=SHADE_REF, lw=0, zorder=0)
        ax.axvspan(*LATE,  color=SHADE_LATE, lw=0, zorder=0)
        ax.axhline(0, color=ZERO, lw=.9, zorder=1)
        ax.grid(axis='y', color=GRID, lw=.65, alpha=.75, zorder=0)

        labels = []
        for k, cut in enumerate(KEEP[kind]):
            c, lw = COL[kind][k], LW[k]
            sm = {m: smooth(series(kind, cut, ssp, m)) for m in MODELS}
            if STYLE == 'model':
                for m in MODELS:
                    ax.plot(sm[m].index, sm[m].values, color=c, ls=LS[m], lw=lw,
                            solid_capstyle='round', dash_capstyle='round', zorder=4 + k)
                y_end = np.nanmean([sm[m].dropna().iloc[-1] for m in MODELS])
            else:
                M = pd.concat(sm, axis=1)
                ax.fill_between(M.index, M.min(1), M.max(1), color=c, alpha=.16, lw=0, zorder=2)
                mu = M.mean(1)
                ax.plot(mu.index, mu.values, color=c, lw=lw + .3, zorder=4 + k)
                y_end = mu.dropna().iloc[-1]
            labels.append((y_end, c, short_label(kind, cut)))
        if END_LABELS:
            end_labels(ax, labels, ylo, yhi)

        ax.set_xlim(START, END); ax.set_ylim(ylo, yhi)
        ax.set_xticks([2021, 2040, 2060, 2080, 2100])
        step = 10 if (yhi - ylo) >= 30 else 5
        ax.set_yticks(np.arange(np.ceil(ylo / step) * step, yhi + .1, step))
        ax.tick_params(labelsize=10, colors=INK, length=4, pad=4)
        for sp in ('top', 'right'):
            ax.spines[sp].set_visible(False)
        if i == 0:
            ax.set_title(SLAB[ssp], fontsize=14, fontweight='bold', color=INK, pad=12)
            ax.tick_params(labelbottom=False)
            ytxt = yhi - .045 * (yhi - ylo)
            ax.text(np.mean(REF_Y), ytxt, 'reference', ha='center', va='top',
                    fontsize=8.5, color='#777777', zorder=5)
            ax.text(np.mean(LATE), ytxt, f'{LATE[0]}–{LATE[1]}', ha='center', va='top',
                    fontsize=8.5, color='#777777', zorder=5)
        else:
            ax.set_xlabel('Year', fontsize=11.5, color=INK, labelpad=6)
        if j == 1:
            ax.tick_params(labelleft=False)

    p = axes[i, 0].get_position()
    fig.text(.053, p.y0 + p.height / 2, VLAB[kind], rotation=90, va='center',
             ha='center', fontsize=11.5, fontweight='bold', color=INK)

# Title, subtitle, shared y label
ylab = ('Change in annual threshold-exceedance days (%)' if VALUE == 'pct'
        else r'Change in annual threshold-exceedance days (d yr$^{-1}$)')
fig.text(.020, .49, ylab, rotation=90, va='center', ha='center', fontsize=11, color=INK)
fig.text(.060, .952, 'a', fontsize=17, fontweight='bold', color=INK)
fig.text(.115, .952, 'Projected change in annual threshold-exceedance days',
         fontsize=17, fontweight='bold', color=INK)
sub = (f"Relative to each model–scenario's own {REF_Y[0]}–{REF_Y[1]} mean; "
       f"five regions combined using traffic weights · {ROLL}-year centred running mean")
if STYLE == 'band':
    sub += ' · line = two-model mean, band = inter-model range'
fig.text(.115, .918, sub, fontsize=10.5, color=INK2)

# Right-hand legend
def side_legend(handles, title, y, **kw):
    lg = fig.legend(handles=handles, title=title, loc='upper left',
                    bbox_to_anchor=(.835, y), frameon=False, fontsize=9.9,
                    title_fontsize=10.5, borderaxespad=0, **kw)
    lg._legend_box.align = 'left'
    fig.add_artist(lg)

if STYLE == 'model':
    side_legend([Line2D([0], [0], color='#222222', lw=2.1, ls=LS[m], label=MLAB[m])
                 for m in MODELS], 'Climate model', .905, handlelength=3.1, handletextpad=.8)
ref_txt = f'({REF_Y[0]}–{REF_Y[1]} mean)'
side_legend([Line2D([0], [0], color=COL['wave'][k], lw=LW[k] + .4, label=thr_label('wave', c))
             for k, c in enumerate(KEEP['wave'])], f'Wave threshold   {ref_txt}', .735,
            handlelength=2.9, handletextpad=.9, labelspacing=.85)
side_legend([Line2D([0], [0], color=COL['wind'][k], lw=LW[k] + .4, label=thr_label('wind', c))
             for k, c in enumerate(KEEP['wind'])], f'Wind threshold   {ref_txt}', .355,
            handlelength=2.9, handletextpad=.9, labelspacing=.85)

fig.text(.115, .055,
         'Line width increases with threshold severity. Baseline days are the mean of the four '
         f'model–scenario baselines. Edge means use at least {MIN_PER} annual values.',
         fontsize=8.5, color=INK2)
plt.show()


In [ ]:
# ============================================================
# C4: annual "days per bin" table (mutually exclusive intervals, sum = valid days in that year)
#   traffic-weighted over five regions; output C4_hazard_bins_annual.csv
#   columns: year, variable, model, scenario, bin, lo, hi, days, n_valid
# ============================================================
import os, numpy as np, pandas as pd, xarray as xr

O   = rf'{DATA_ROOT}/new_climate/_proj_main'
TB  = os.path.join(O, 'tables')
M2C = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid2'
OUT_CSV = os.path.join(TB, 'C4_hazard_bins_annual.csv')

EDGES  = {'wave': [0, 1.25, 2.5, 4.0, 6.0, 9.0, np.inf],
          'wind': [0, 8.0, 10.8, 13.9, 17.2, 20.8, np.inf]}
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']
REG = {'NW_Europe':   [(36,45,-15,-6),(45,63,-15,-6),(43,63,-6,0),(48,63,0,13)],
       'NW_Atlantic': [(18,31,-98,-60),(31,42,-82,-60),(42,50,-67,-60),(45,55,-60,-46)],
       'Med_Black':   [(30,43,-6,0),(30,46,0,19),(30,46,19,26.5),(30,41,26.5,37),(41,47.5,26.5,42)],
       'Asia':        [(-10,20,95,130),(20,46,117,146)],
       'Indian_Ocean_N': [(5,30,50,78),(5,26,78,95)]}

cells = xr.open_dataarray(os.path.join(O, 'obs_wave.nc')).cell.values
t0 = pd.to_datetime(xr.open_dataarray(os.path.join(O, 'cm_wave_ACCESS_his.nc')).time.values)
k0 = (t0.year >= 2002) & (t0.year <= 2014)
GOOD = np.ones(len(cells), bool)
for m in MODELS:
    for kd in ('wave', 'wind'):
        GOOD &= np.isfinite(xr.open_dataarray(
            os.path.join(O, f'cm_{kd}_{m}_his.nc')).load().values[k0]).mean(0) >= 0.9

def reg_of(g):
    la, lo = map(float, g.split('_'))
    for r, bs in REG.items():
        for S, N, W, E in bs:
            if S <= la < N and W <= lo < E:
                return r
    return None
RG = np.array([reg_of(g) for g in cells])
pg = pd.read_parquet(os.path.join(M2C, 'panel_global.parquet'), columns=['region', 'grid_id', 'E_it'])
pg = pg[pg.grid_id.isin(set(cells[GOOD]))]
WR = pg.groupby('region').E_it.sum(); WR = WR / WR.sum()

def annual_bins(arr, times, kind):
    yr = times.year.values; u = np.unique(yr); idx = np.searchsorted(yr, u)
    ok = np.isfinite(arr); e = EDGES[kind]
    def agg(mask):                      # cell counts → region mean → traffic-weighted
        cnt = np.add.reduceat(mask.astype(np.float32), idx, 0)
        return sum(cnt[:, (RG == r) & GOOD].mean(1) * WR[r] for r in REG)
    nval = agg(ok); rows = []
    for b in range(len(e) - 1):
        lo, hi = e[b], e[b + 1]
        m = ok & (arr >= lo) & ((arr < hi) if np.isfinite(hi) else True)
        name = f'{lo:g}–{hi:g}' if np.isfinite(hi) else f'≥{lo:g}'
        rows.append(pd.DataFrame({'year': u, 'bin': name, 'lo': lo, 'hi': hi,
                                  'days': agg(m), 'n_valid': nval}))
    return pd.concat(rows, ignore_index=True)

rows = []
for kind in ('wave', 'wind'):
    for m in MODELS:
        for ssp in SSPS:
            q = xr.open_dataarray(os.path.join(O, f'qdm_{kind}_{m}_{ssp}.nc')).load()
            d = annual_bins(q.values, pd.to_datetime(q.time.values), kind)
            d['variable'] = kind; d['model'] = m; d['scenario'] = ssp
            rows.append(d); del q
            print(f'  {kind} {m} {ssp} ok', flush=True)
C4 = pd.concat(rows, ignore_index=True)
C4.to_csv(OUT_CSV, index=False)
s = C4.groupby(['variable', 'model', 'scenario', 'year']).agg(days=('days', 'sum'),
                                                              nv=('n_valid', 'first'))
print('C4', C4.shape, '->', OUT_CSV, '| max bin-closure error', round(float((s.days - s.nv).abs().max()), 6))
print(C4[(C4.year == 2030) & (C4.model == 'ACCESS') & (C4.scenario == 'ssp585')]
      .pivot(index='bin', columns='variable', values='days').round(1))


In [ ]:
# ============================================================
# Stacked area plot of days per bin: each year's 365 days split by threshold interval, stack total = valid days
#   one variable per figure: rows = scenario, columns = model
# Input: C4_hazard_bins_annual.csv (produced by the previous cell)
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Patch

CSV = rf'{DATA_ROOT}/new_climate/_proj_main/tables/C4_hazard_bins_annual.csv'
C4  = pd.read_csv(CSV)

# ── Tunable parameters ────────────────────────────────────────────────
ROLL       = 10                 # centred moving average (years); 1 = no smoothing
START, END = 2021, 2100
REF_Y      = (2021, 2040)       # reference period, dashed line at its end
Y0         = 0                  # y-axis lower limit; set 200 / 250 to zoom on upper layers (calm days clipped)
MODE       = 'days'             # 'days' = days/yr  |  'share' = % of valid days
VARS       = ['wave', 'wind']   # one figure per variable
MODELS = ['ACCESS', 'ECEARTH']
SSPS   = ['ssp126', 'ssp585']
MERGE_TOP = {'wave': 6.0, 'wind': 17.2}   # bins ≥ this threshold merged into one layer; None = no merging

# Colours: bottom layer (calm days) light grey, darker shades of one hue upward, top layer (extreme) in a contrasting colour
COL = {'wave': ['#e4e9ee', '#c9dff0', '#8ec6e6', '#3a94cd', '#2b6cb0'],
       'wind': ['#e4e9ee', '#fbe3bf', '#f6c053', '#ee7f24', "#f76f00"]}

UNIT  = {'wave': 'm', 'wind': 'm s$^{-1}$'}
SYM   = {'wave': '$H_s$', 'wind': '$U_{10}$'}
VLAB  = {'wave': 'Significant wave height', 'wind': '10-m wind speed'}
MLAB  = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
SLAB  = {'ssp126': 'SSP1–2.6', 'ssp585': 'SSP5–8.5'}
INK, INK2 = '#1a1a1a', '#5c6166'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False, 'mathtext.fontset': 'dejavusans',
                     'axes.linewidth': 0.85})

def bin_label(kind, lo, hi):
    if lo == 0:
        return f'{SYM[kind]} < {hi:g} {UNIT[kind]}'
    if np.isfinite(hi):
        return f'{lo:g}–{hi:g} {UNIT[kind]}'
    return f'≥ {lo:g} {UNIT[kind]}'

def stack_matrix(kind, model, ssp):
    """→ years, (n_bins, n_years) matrix (bottom to top), list of bins"""
    d = C4[(C4.variable == kind) & (C4.model == model) & (C4.scenario == ssp)]
    p = d.pivot(index='year', columns='lo', values='days').sort_index()
    p = p[sorted(p.columns)]                                   # low thresholds at the bottom
    top = MERGE_TOP.get(kind) if MERGE_TOP else None
    if top is not None:                                        # bins ≥top merged into one layer
        hi_cols = [c for c in p.columns if c >= top]
        p[top] = p[hi_cols].sum(axis=1)
        p = p.drop(columns=[c for c in hi_cols if c != top])
        p = p[sorted(p.columns)]
    if MODE == 'share':
        nv = d.groupby('year').n_valid.first().reindex(p.index)
        p = p.div(nv, axis=0) * 100
    if ROLL > 1:
        p = p.rolling(ROLL, center=True, min_periods=ROLL // 2).mean()
    p = p.loc[(p.index >= START) & (p.index <= END)].dropna()
    bins = [(lo, (np.inf if (top is not None and lo >= top) else d[d.lo == lo].hi.iloc[0]))
            for lo in p.columns]
    return p.index.values, p.values.T, bins
def plot_var(kind, letter=None):
    fig, axes = plt.subplots(len(SSPS), len(MODELS), figsize=(12.5, 6.4), dpi=500,
                             sharex=True, sharey=True)
    fig.subplots_adjust(left=.08, right=.80, top=.84, bottom=.10, wspace=.08, hspace=.22)
    for i, ssp in enumerate(SSPS):
        for j, model in enumerate(MODELS):
            ax = axes[i, j]
            yrs, M, bins = stack_matrix(kind, model, ssp)
            ax.stackplot(yrs, M, colors=COL[kind][:len(bins)], lw=0.4,
                         edgecolor='white', zorder=2)
            ax.axvline(REF_Y[1], color='#333333', lw=.8, ls=(0, (2, 2)), zorder=3)
            ax.set_xlim(START, END)
            ax.set_ylim(Y0, 100 if MODE == 'share' else None)
            ax.set_xticks([2021, 2040, 2060, 2080, 2100])
            ax.tick_params(labelsize=9.5, colors=INK, length=3.5)
            for sp in ('top', 'right'):
                ax.spines[sp].set_visible(False)
            if i == 0:
                ax.set_title(MLAB[model], fontsize=12, fontweight='bold', color=INK, pad=8)
            if j == 0:
                ax.set_ylabel(f'{SLAB[ssp]}\n' + ('share of days (%)' if MODE == 'share'
                              else 'days per year'), fontsize=10, color=INK)
            if i == len(SSPS) - 1:
                ax.set_xlabel('Year', fontsize=10.5, color=INK)
    if MODE == 'days' and Y0 == 0:
        axes[0, 0].set_ylim(0, 370)

    if letter:
        fig.text(.02, .93, letter, fontsize=15, fontweight='bold', color=INK)
    fig.text(.08, .93, f'{VLAB[kind]}: composition of the year by threshold band',
             fontsize=13.5, fontweight='bold', color=INK)
    fig.text(.08, .885, ('Bias-corrected daily maxima · five regions combined using traffic '
                         f'weights · {ROLL}-year centred running mean · dotted line = end of '
                         f'{REF_Y[0]}–{REF_Y[1]} reference'),
             fontsize=8.5, color=INK2)

    # legend (top to bottom = extreme to calm, matching stack order)
    _, _, bins = stack_matrix(kind, MODELS[0], SSPS[0])
    handles = [Patch(facecolor=COL[kind][k], edgecolor='none', label=bin_label(kind, lo, hi))
               for k, (lo, hi) in enumerate(bins)][::-1]
    lg = fig.legend(handles=handles, title='Threshold band', loc='upper left',
                    bbox_to_anchor=(.815, .80), frameon=False, fontsize=9.5,
                    title_fontsize=10, labelspacing=.8, borderaxespad=0)
    lg._legend_box.align = 'left'
    plt.show()

for n, kind in enumerate(VARS):
    plot_var(kind, letter='ab'[n])
